# 04 - Qdrant Ingestion (Hue Foods RAG MVP)

Notebook này trình bày Phase 4 và **kiểm tra read-only** collection Qdrant thật đang chạy. Run All **không** chạy ingestion, không upsert, không reset, không delete - chỉ đọc metadata, schema và payload projection an toàn.

**Prerequisite**

- Qdrant local đang chạy qua Docker Compose (image pinned v1.18.3, port 6333).
- Collection `hue_foods_e5_small_384` đã được ingestion tạo trước đó với đúng 572 points.

**Kết quả mong đợi khi Run All**

- Collection tồn tại, schema khớp: dense 384 cosine + sparse index enabled.
- Exact count = 572 points.
- Payload mẫu chỉ in các field metadata đã phê duyệt, không in toàn bộ text.

Nếu Qdrant tắt, collection thiếu, schema lệch hoặc count khác 572, notebook fail rõ ràng - đó là hành vi mong muốn.


In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Khong tim thay thu muc backend/. Hay mo notebook nay tu repo root "
        "hoac tu thu muc notebooks/."
    )
print(f"backend on path: {sys.path[0]}")


## Cấu hình vector database (`settings.yaml`)

Nhóm `vector_database` khai báo Qdrant local: URL, active collection `hue_foods_e5_small_384`, `reset_collection: false` (ingestion không bao giờ xóa), dimension 384, cosine, timeout 30 giây.


In [ ]:
from core.settings_loader import load_settings

settings = load_settings()
db = settings["vector_database"]
print("url:", db["url"])
print("collection_name:", db["collection_name"])
print("reset_collection:", db["reset_collection"])
print("vector_size:", db["vector_size"])
print("distance:", db["distance"])
print("timeout:", db["timeout"])


## Point contract và UUID5 deterministic

Mỗi point ID là `uuid.uuid5(uuid.NAMESPACE_URL, f"hue-rag:{chunk_id}")`. Cùng một `chunk_id` luôn ra cùng một ID nên upsert là idempotent; original `chunk_id` được giữ trong payload.


In [ ]:
from vectorstore.hybrid_index import point_id_for

chunk_id = "foods/restaurants/example.md|Tóm tắt|0"
first = point_id_for(chunk_id)
print("point id:", first)
print("deterministic:", point_id_for(chunk_id) == first)
print("different chunk:", point_id_for("foods/cafes/other.md|Tóm tắt|0") != first)


## Schema kỳ vọng của collection

`expected_schema` mô tả named vectors: `dense` (size 384, cosine) và `sparse` (SparseVectorParams với index được bật). Runtime chỉ tạo collection khi absent; collection đã tồn tại phải khớp schema nếu không pipeline fail closed.


In [ ]:
from vectorstore.qdrant import expected_schema

schema = expected_schema(settings)
print("vector names:", sorted(schema))
print("dense size:", schema["dense"].size)
print("dense distance:", schema["dense"].distance)
print("sparse index enabled:", schema["sparse"].index is not None)


## Kiểm tra read-only collection thật

Cell dưới kết nối Qdrant thật theo settings hiện hành và chỉ thực hiện các thao tác read-only: `collection_exists`, `get_collection`, `validate_collection_info` (fail fast khi schema lệch), `count` exact và `scroll` hai payload với `with_vectors=False`. Payload chỉ in các field metadata đã phê duyệt và độ dài text - không in toàn bộ nội dung chunk.


In [ ]:
from vectorstore.qdrant import client_from_settings, validate_collection_info

client = client_from_settings(settings)
name = settings["vector_database"]["collection_name"]

if not client.collection_exists(name):
    raise RuntimeError(
        f"Collection {name} khong ton tai. Hay bat Qdrant bang Docker Compose "
        "va chay ingestion da duoc phe duyet truoc khi chay notebook nay."
    )

info = client.get_collection(name)
validate_collection_info(info, settings)  # fail fast khi schema lech

params = info.config.params
count = client.count(name, exact=True).count
if count != 572:
    raise RuntimeError(f"Expected 572 points, found {count}")

print("collection:", name)
print("status:", info.status)
print("dense:", params.vectors["dense"].size, params.vectors["dense"].distance)
print("sparse index enabled:", params.sparse_vectors["sparse"].index is not None)
print("point count (exact):", count)

records, _ = client.scroll(name, limit=2, with_payload=True, with_vectors=False)
APPROVED_FIELDS = (
    "chunk_id", "source", "title", "section", "category",
    "subcategory", "chunk_type", "embedding_model", "embedding_dimension",
)
for record in records:
    payload = record.payload
    print({key: payload.get(key) for key in APPROVED_FIELDS})
    print("  text length:", len(payload.get("text", "")))


## Checklist xác nhận Phase 4

1. Config hiển thị đúng collection name, `reset_collection: false`.
2. `point_id_for` deterministic cho cùng chunk_id.
3. Schema kỳ vọng đúng dense 384 cosine + sparse index.
4. Collection thật tồn tại, schema khớp, status green, exact count = 572.
5. Payload mẫu chỉ in approved metadata fields + độ dài text.
6. Không có thao tác mutation nào trong notebook này.
